# Profilage DVF 2022 — Étape 2b : lignes rendues identiques par l'expurgation

Le fichier DVF représente chaque bien concerné par une mutation sur une ligne
distincte. La notice descriptive (DGFiP, 2022, p. 3) le confirme : « Quand une
disposition comporte plusieurs locaux ou plusieurs natures de culture, le fichier
de restitution comporte autant de lignes qu'il y a de locaux ». Si une vente
porte sur deux garages, le fichier contient deux lignes. Ces deux garages sont
des biens physiquement différents.

Cependant, la colonne Identifiant local — qui distinguait chaque bien de manière
unique — a été supprimée lors de la publication en open data (décret 2018-1350).
Sans cet identifiant, deux garages identiques dans le même immeuble, vendus
dans la même mutation, deviennent des **lignes indistinguables** : même date,
même prix, même commune, même surface, même type.

Ce notebook identifie ces lignes rendues identiques, mesure leur ampleur et
caractérise leur profil. L'objectif n'est pas de les « supprimer » (ce serait
supprimer des biens réels), mais de comprendre et documenter le phénomène
pour guider les décisions du data steward.

## Cellule 1 — Installation

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
print("DuckDB prêt.")

DuckDB prêt.


## Cellule 2 — Réglages

In [2]:
import duckdb
import os
from pathlib import Path

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
print(f"Fichier : {FICHIER.name}")

Fichier : dvf-2022.parquet


## Cellule 3 — Préparation des données de base

Recalculer le nombre de lignes, la liste des colonnes et identifier les
colonnes entièrement vides. La variable `cols_sql` (concaténation des
35 noms de colonnes exploitables) sert de critère de comparaison pour
détecter les lignes identiques : deux lignes sont indistinguables si et
seulement si elles partagent les mêmes valeurs sur ces 35 colonnes.

Les 8 colonnes vides sont exclues du GROUP BY : puisqu'elles sont NULL
partout, elles ne contribuent pas à distinguer les lignes.

In [3]:
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
colonnes = con.execute(f"DESCRIBE SELECT * FROM '{pq}'").fetchall()
col_names = [c[0] for c in colonnes]

# Identifier les colonnes 100 % vides
cols_vides = []
for c in col_names:
    n_null = con.execute(f'SELECT count(*) FROM \'{pq}\' WHERE "{c}" IS NULL').fetchone()[0]
    if n_null == nb_lignes:
        cols_vides.append(c)

cols_exploitables = [c for c in col_names if c not in cols_vides]
cols_sql = ", ".join([f'"' + c + '"' for c in cols_exploitables])

print(f"Lignes : {nb_lignes:,}".replace(",", " "))
print(f"Colonnes exploitables : {len(cols_exploitables)}")
print(f"Colonnes vides (exclues) : {len(cols_vides)}")

Lignes : 4 617 590
Colonnes exploitables : 35
Colonnes vides (exclues) : 8


## Cellule 4 — Combien de lignes sont rendues identiques ?

Regrouper les lignes par les 35 colonnes exploitables. Si un groupe contient
plus d'une ligne, ces lignes sont indistinguables entre elles. Le compteur
« lignes sans identifiant unique » mesure, pour chaque groupe de taille n,
les (n − 1) lignes qui ne peuvent plus être distinguées de la première.

Important : ces lignes représentent des biens physiquement différents.
La notice descriptive (DGFiP, 2022, p. 3) confirme que le fichier comporte
« autant de lignes qu'il y a de locaux ». La suppression de l'Identifiant
local (décret 2018-1350) a rendu ces biens distincts indistinguables.

In [4]:
dup = con.execute(f"""
    SELECT
        count(*) AS nb_groupes,
        sum(n)   AS lignes_total,
        sum(n-1) AS lignes_sans_identifiant_unique
    FROM (
        SELECT count(*) AS n
        FROM '{pq}'
        GROUP BY {cols_sql}
        HAVING count(*) > 1
    )
""").fetchone()

print("Lignes rendues identiques par l'expurgation")
print("=" * 55)
print(f"  Groupes de lignes identiques       : {dup[0]:>10,}".replace(",", " "))
print(f"  Lignes dans ces groupes            : {dup[1]:>10,}".replace(",", " "))
print(f"  Dont sans identifiant unique       : {dup[2]:>10,}".replace(",", " "))
print(f"  Taux                               : {100*dup[2]/nb_lignes:.2f} %")
print()
print("Ces lignes représentent des biens physiquement différents")
print("(deux garages distincts, par exemple) qui sont devenus")
print("indistinguables après la suppression de l'Identifiant local.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lignes rendues identiques par l'expurgation
  Groupes de lignes identiques       :    191 468
  Lignes dans ces groupes            :    536 750
  Dont sans identifiant unique       :    345 282
  Taux                               : 7.48 %

Ces lignes représentent des biens physiquement différents
(deux garages distincts, par exemple) qui sont devenus
indistinguables après la suppression de l'Identifiant local.


## Cellule 5 — Distribution de la taille des groupes

Un groupe de 2 lignes identiques pourrait résulter d'une coïncidence. Un groupe
de 50 ou 100 lignes ne peut pas l'être : il reflète un mécanisme structurel.
La distribution de la taille des groupes permet de distinguer ces deux scénarios.

In [5]:
taille_groupes = con.execute(f"""
    SELECT
        n AS taille_groupe,
        count(*) AS nb_groupes,
        count(*) * n AS nb_lignes_concernees
    FROM (
        SELECT count(*) AS n
        FROM '{pq}'
        GROUP BY {cols_sql}
        HAVING count(*) > 1
    )
    GROUP BY n
    ORDER BY n
""").fetchall()

print(f"{'Taille du groupe':<20} {'Nb groupes':>12} {'Lignes concernées':>20}")
print("-" * 55)
for taille, nb_gr, nb_lig in taille_groupes:
    print(f"  {taille:<20} {nb_gr:>10,} {nb_lig:>18,}".replace(",", " "))
print()
print(f"La distribution s'étend de 2 à {taille_groupes[-1][0]}.")
print(f"Les paires dominent ({taille_groupes[0][1]:,} groupes, "
      f"{100*taille_groupes[0][1]/dup[0]:.0f} % des groupes),".replace(",", " "))
print(f"mais la queue de distribution est longue.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Taille du groupe       Nb groupes    Lignes concernées
-------------------------------------------------------
  2                       143 932            287 864
  3                        27 791             83 373
  4                         9 304             37 216
  5                         3 071             15 355
  6                         1 984             11 904
  7                           876              6 132
  8                           837              6 696
  9                           560              5 040
  10                          458              4 580
  11                          269              2 959
  12                          284              3 408
  13                          166              2 158
  14                          188              2 632
  15                          143              2 145
  16                          174              2 784
  17                           94              1 598
  18                          123        

## Cellule 6 — Exemples concrets de groupes de lignes identiques

Descendre au niveau des lignes individuelles pour voir à quoi ressemblent
concrètement les lignes rendues identiques. Trois exemples sont extraits :
un groupe de 2, un groupe de 5 et le plus grand groupe.

La requête utilise GROUP BY (et non LIMIT sur les lignes brutes) pour
garantir que les valeurs affichées correspondent bien à un groupe de
lignes strictement identiques sur les 35 colonnes.

In [6]:
cols_aff = ["Date mutation", "Nature mutation", "Valeur fonciere",
            "Code departement", "Commune", "Type local",
            "Surface reelle bati", "Nombre pieces principales",
            "Surface terrain", "Nature culture", "Nombre de lots"]

def afficher_groupe(label, df, n):
    """Affiche les colonnes principales d'un groupe de lignes identiques."""
    print(f"{label} ({n} lignes identiques) :")
    print()
    for c in cols_aff:
        if c in df.columns:
            val = df[c].iloc[0]
            if val is None:
                val = "(vide)"
            print(f"  {c:<30} {val}")
    print()
    print(f"  Ces {n} lignes représentent {n} biens distincts rendus")
    print(f"  indistinguables par la suppression de l'Identifiant local.")
    print()

# --- Groupe de 2 ---
groupe2 = con.execute(f"""
    SELECT {cols_sql}
    FROM '{pq}'
    GROUP BY {cols_sql}
    HAVING count(*) = 2
    ORDER BY "Date mutation", "Commune"
    LIMIT 1
""").fetchdf()
afficher_groupe("Exemple 1 — Groupe de 2", groupe2, 2)

# --- Groupe de 5 ---
groupe5 = con.execute(f"""
    SELECT {cols_sql}
    FROM '{pq}'
    GROUP BY {cols_sql}
    HAVING count(*) = 5
    ORDER BY "Date mutation", "Commune"
    LIMIT 1
""").fetchdf()
afficher_groupe("Exemple 2 — Groupe de 5", groupe5, 5)

# --- Plus grand groupe ---
max_n = con.execute(f"""
    SELECT max(n) FROM (
        SELECT count(*) AS n FROM '{pq}' GROUP BY {cols_sql} HAVING count(*) > 1
    )
""").fetchone()[0]

plus_grand = con.execute(f"""
    SELECT {cols_sql}
    FROM '{pq}'
    GROUP BY {cols_sql}
    HAVING count(*) = {max_n}
    LIMIT 1
""").fetchdf()
afficher_groupe(f"Exemple 3 — Plus grand groupe", plus_grand, max_n)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exemple 1 — Groupe de 2 (2 lignes identiques) :

  Date mutation                  2022-01-03 00:00:00
  Nature mutation                Vente
  Valeur fonciere                96700
  Code departement               34
  Commune                        AGDE
  Type local                     Dépendance
  Surface reelle bati            0
  Nombre pieces principales      0
  Surface terrain                <NA>
  Nature culture                 (vide)
  Nombre de lots                 1

  Ces 2 lignes représentent 2 biens distincts rendus
  indistinguables par la suppression de l'Identifiant local.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exemple 2 — Groupe de 5 (5 lignes identiques) :

  Date mutation                  2022-01-03 00:00:00
  Nature mutation                Vente
  Valeur fonciere                75000
  Code departement               35
  Commune                        GUIPRY-MESSAC
  Type local                     Dépendance
  Surface reelle bati            0
  Nombre pieces principales      0
  Surface terrain                1000
  Nature culture                 S
  Nombre de lots                 0

  Ces 5 lignes représentent 5 biens distincts rendus
  indistinguables par la suppression de l'Identifiant local.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exemple 3 — Plus grand groupe (976 lignes identiques) :

  Date mutation                  2022-12-27 00:00:00
  Nature mutation                Vente
  Valeur fonciere                314985152
  Code departement               56
  Commune                        VANNES
  Type local                     Dépendance
  Surface reelle bati            0
  Nombre pieces principales      0
  Surface terrain                52195
  Nature culture                 S
  Nombre de lots                 0

  Ces 976 lignes représentent 976 biens distincts rendus
  indistinguables par la suppression de l'Identifiant local.



## Cellule 7 — Profil des lignes indistinguables : type de bien

Les lignes indistinguables sont-elles réparties uniformément entre les types de
biens, ou certains types sont-ils massivement sur-représentés ? Comparer la
répartition dans les groupes de lignes identiques avec celle du fichier complet.

In [7]:
profil_type = con.execute(f"""
    WITH lignes_indist AS (
        SELECT *, count(*) OVER (PARTITION BY {cols_sql}) AS n_copies
        FROM '{pq}'
    )
    SELECT
        CASE
            WHEN "Type local" IS NULL THEN '(vide)'
            ELSE "Type local"
        END AS type_local,
        count(*) AS nb_dans_indist,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_indist
    FROM lignes_indist
    WHERE n_copies > 1
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

profil_type_global = con.execute(f"""
    SELECT
        CASE
            WHEN "Type local" IS NULL THEN '(vide)'
            ELSE "Type local"
        END AS type_local,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_global
    FROM '{pq}'
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

print("Répartition des lignes indistinguables par type de bien :")
print(profil_type.to_string(index=False))
print()
print("Pour comparaison — répartition dans le fichier complet :")
print(profil_type_global.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Répartition des lignes indistinguables par type de bien :
                              type_local  nb_dans_indist  pct_indist
                              Dépendance          422496        78.7
                             Appartement           72477        13.5
                                  (vide)           26134         4.9
                                  Maison            9544         1.8
Local industriel. commercial ou assimilé            6099         1.1

Pour comparaison — répartition dans le fichier complet :
                              type_local  pct_global
                                  (vide)        40.6
                              Dépendance        26.1
                                  Maison        16.4
                             Appartement        13.8
Local industriel. commercial ou assimilé         3.1


## Cellule 8 — Profil des lignes indistinguables : nature de la mutation

Même logique de comparaison pour la nature de la mutation (Vente, VEFA,
Échange, etc.).

In [8]:
profil_nat = con.execute(f"""
    WITH lignes_indist AS (
        SELECT *, count(*) OVER (PARTITION BY {cols_sql}) AS n_copies
        FROM '{pq}'
    )
    SELECT
        "Nature mutation",
        count(*) AS nb_dans_indist,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_indist
    FROM lignes_indist
    WHERE n_copies > 1
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

profil_nat_global = con.execute(f"""
    SELECT
        "Nature mutation",
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_global
    FROM '{pq}'
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

print("Répartition des lignes indistinguables par nature de mutation :")
print(profil_nat.to_string(index=False))
print()
print("Pour comparaison — répartition dans le fichier complet :")
print(profil_nat_global.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Répartition des lignes indistinguables par nature de mutation :
                   Nature mutation  nb_dans_indist  pct_indist
                             Vente          532076        99.1
                           Echange            1888         0.4
                      Adjudication            1345         0.3
Vente en l'état futur d'achèvement            1062         0.2
             Vente terrain à bâtir             321         0.1
                     Expropriation              58         0.0

Pour comparaison — répartition dans le fichier complet :
                   Nature mutation  pct_global
                             Vente        92.4
Vente en l'état futur d'achèvement         6.1
                           Echange         1.0
             Vente terrain à bâtir         0.3
                      Adjudication         0.2
                     Expropriation         0.0


## Cellule 9 — Profil des lignes indistinguables : nombre de lots

Le nombre de lots indique combien de fractions de copropriété sont associées
à la transaction. Les transactions sans lot (0 lot) devraient être
sur-représentées : sans numéro de lot pour différencier les lignes,
l'indistinguabilité est plus probable.

In [9]:
profil_lots = con.execute(f"""
    WITH lignes_indist AS (
        SELECT *, count(*) OVER (PARTITION BY {cols_sql}) AS n_copies
        FROM '{pq}'
    )
    SELECT
        "Nombre de lots",
        count(*) AS nb_dans_indist,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_indist
    FROM lignes_indist
    WHERE n_copies > 1
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").fetchdf()

profil_lots_global = con.execute(f"""
    SELECT
        "Nombre de lots",
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_global
    FROM '{pq}'
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").fetchdf()

print("Répartition des lignes indistinguables par nombre de lots (top 10) :")
print(profil_lots.to_string(index=False))
print()
print("Pour comparaison — répartition dans le fichier complet :")
print(profil_lots_global.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Répartition des lignes indistinguables par nombre de lots (top 10) :
 Nombre de lots  nb_dans_indist  pct_indist
              0          426259        79.4
              2           44794         8.3
              1           30901         5.8
              3           24092         4.5
              4            6474         1.2
              5            1978         0.4
              6            1089         0.2
              7             485         0.1
              8             249         0.0
              9             160         0.0

Pour comparaison — répartition dans le fichier complet :
 Nombre de lots  pct_global
              0        67.6
              1        22.5
              2         8.1
              3         1.2
              4         0.3
              5         0.1
              6         0.1
              7         0.0
              8         0.0
              9         0.0


## Cellule 10 — Colonnes vides et indistinguabilité

Les 8 colonnes supprimées (identifiants de document, de local, articles CGI)
contenaient-elles l'information nécessaire pour distinguer les lignes
actuellement identiques ? Par définition, les lignes d'un groupe sont
identiques sur les 35 colonnes restantes. La colonne Identifiant local
aurait ajouté une valeur différente à chaque ligne du groupe.

On vérifie aussi combien de groupes ont une valeur foncière renseignée,
pour quantifier le poids des transactions sans prix.

In [10]:
test_mutation = con.execute(f"""
    WITH groupes AS (
        SELECT
            {cols_sql},
            count(*) AS n
        FROM '{pq}'
        GROUP BY {cols_sql}
        HAVING count(*) > 1
    )
    SELECT
        count(*) AS nb_groupes_total,
        sum(CASE WHEN "Valeur fonciere" IS NOT NULL THEN 1 ELSE 0 END) AS avec_vf,
        sum(CASE WHEN "Valeur fonciere" IS NULL THEN 1 ELSE 0 END) AS sans_vf
    FROM groupes
""").fetchone()

print("Groupes de lignes identiques et valeur foncière")
print("=" * 55)
print(f"  Groupes total          : {test_mutation[0]:>10,}".replace(",", " "))
print(f"  Avec valeur foncière   : {test_mutation[1]:>10,}".replace(",", " "))
print(f"  Sans valeur foncière   : {test_mutation[2]:>10,}".replace(",", " "))
print()
print("Chaque groupe contient, par définition, des lignes avec la même")
print("date, la même commune, la même valeur foncière, le même type")
print("local, la même surface — sur toutes les colonnes exploitables.")
print("Seuls les identifiants supprimés auraient pu les distinguer.")
print()
print("C'est la tension D17 (Confidentialité) × D4 (Unicité) :")
print("la protection de la vie privée a rendu des biens distincts")
print("indistinguables dans le fichier publié.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Groupes de lignes identiques et valeur foncière
  Groupes total          :    191 468
  Avec valeur foncière   :    190 631
  Sans valeur foncière   :        837

Chaque groupe contient, par définition, des lignes avec la même
date, la même commune, la même valeur foncière, le même type
local, la même surface — sur toutes les colonnes exploitables.
Seuls les identifiants supprimés auraient pu les distinguer.

C'est la tension D17 (Confidentialité) × D4 (Unicité) :
la protection de la vie privée a rendu des biens distincts
indistinguables dans le fichier publié.


## Cellule 11 — Répartition par département (top 10)

Les lignes indistinguables sont-elles réparties uniformément sur le territoire,
ou concentrées dans certains départements ?

In [11]:
dup_dept = con.execute(f"""
    WITH lignes_indist AS (
        SELECT *, count(*) OVER (PARTITION BY {cols_sql}) AS n_copies
        FROM '{pq}'
    )
    SELECT
        "Code departement",
        count(*) AS nb_lignes_indist,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM lignes_indist
    WHERE n_copies > 1
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").fetchdf()

print("Top 10 des départements par nombre de lignes indistinguables :")
print(dup_dept.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Top 10 des départements par nombre de lignes indistinguables :
Code departement  nb_lignes_indist  pct
              56             30471  5.7
              59             15460  2.9
              83             14692  2.7
              22             13833  2.6
              75             12805  2.4
              13             12798  2.4
              69             12676  2.4
              06             12113  2.3
              33             12031  2.2
              42              9575  1.8


## Cellule 12 — Synthèse de l'étape 2b

Récapituler les constats pour faciliter la rédaction et la transition
vers l'étape suivante.

In [12]:
print("Synthèse — Lignes rendues identiques dans DVF 2022")
print("=" * 60)
print(f"  Groupes de lignes identiques      : {dup[0]:>10,}".replace(",", " "))
print(f"  Lignes sans identifiant unique     : {dup[2]:>10,}".replace(",", " "))
print(f"  Taux                               : {100*dup[2]/nb_lignes:.2f} %")
print(f"  Taille maximale d'un groupe        : {taille_groupes[-1][0]}")
print()
print("Répartition par taille de groupe :")
for taille, nb_gr, nb_lig in taille_groupes[:5]:
    print(f"    Groupes de {taille:<5} : {nb_gr:>10,}".replace(",", " "))
print()
print("Profil des lignes indistinguables :")
print(f"  Dépendances : 78,7 % (vs 26,1 % dans le fichier complet)")
print(f"  0 lot       : 79,4 % (vs 67,6 %)")
print(f"  Ventes      : 99,1 % (vs 92,4 %)")
print()
print("Interprétation :")
print("  Ces lignes ne sont pas des erreurs. Chaque ligne représente")
print("  un bien réel (notice descriptive, DGFiP, 2022, p. 3).")
print("  La suppression de l'Identifiant local (décret 2018-1350) a")
print("  rendu ces biens distincts indistinguables.")
print()
print("  C'est la tension D17 (Confidentialité) × D4 (Unicité) :")
print("  la protection de la vie privée a dégradé l'unicité.")
print()
print("Recommandation du data steward :")
print("  (a) Documenter l'indistinguabilité (ne pas supprimer).")
print("  (b) Agréger au niveau de la mutation pour reconstituer")
print("      les transactions.")
print("  (c) Séparer entraînement/test au niveau de la mutation,")
print("      pas de la ligne, pour éviter les fuites de données.")

Synthèse — Lignes rendues identiques dans DVF 2022
  Groupes de lignes identiques      :    191 468
  Lignes sans identifiant unique     :    345 282
  Taux                               : 7.48 %
  Taille maximale d'un groupe        : 976

Répartition par taille de groupe :
    Groupes de 2     :    143 932
    Groupes de 3     :     27 791
    Groupes de 4     :      9 304
    Groupes de 5     :      3 071
    Groupes de 6     :      1 984

Profil des lignes indistinguables :
  Dépendances : 78,7 % (vs 26,1 % dans le fichier complet)
  0 lot       : 79,4 % (vs 67,6 %)
  Ventes      : 99,1 % (vs 92,4 %)

Interprétation :
  Ces lignes ne sont pas des erreurs. Chaque ligne représente
  un bien réel (notice descriptive, DGFiP, 2022, p. 3).
  La suppression de l'Identifiant local (décret 2018-1350) a
  rendu ces biens distincts indistinguables.

  C'est la tension D17 (Confidentialité) × D4 (Unicité) :
  la protection de la vie privée a dégradé l'unicité.

Recommandation du data steward :


## Cellule 13 — Fermeture

In [13]:
con.close()
print("Connexion DuckDB fermée.")

Connexion DuckDB fermée.
